# 통합 관제 시스템 구축하기 
YOLO + OpenAI + TTS

## YOLO 최종

In [37]:
import cv2
import numpy as np
from PIL import Image, ImageDraw, ImageFont
import platform 

#========== 경로 설정 ==========
weights_path = "yolo/yolov3.weights"
config_path = "yolo/yolov3.cfg"
# names_path = "yolo/coco.names"
names_path = "yolo/coco_korean.names" #한글 테스트를 위해 변경

#========== 한글 폰트 설정 ==========
def get_font():
    font_size = 20
    try:
        if platform.system() == "Windows":
            return ImageFont.truetype("malgun.ttf", font_size)
        elif platform.system() == "Darwin":  # macOS
            return ImageFont.truetype("AppleGothic.ttf", font_size)
        else:  # Linux 등
            return ImageFont.load_default()
    except IOError:
        return ImageFont.load_default()

font = get_font()

#========== 라벨(이름) 가져오기 ==========
with open(names_path, "r", encoding='utf-8') as name_file:
    label_list = name_file.read().strip().split("\n")
    # strip : 공백 제거 / split: 줄마다 잘라서 리스트로

#========== 모델 불러오기 ==========
net = cv2.dnn.readNet(weights_path, config_path)
# 밖에 선언하는 이유: 그라디오에서 n초마다 한 번씩 계속 호출될텐데 그때마다 계속 선언하면 메모리 부하될수도

def detect_objects(image_array):
    #========== image array -> CV로 읽기 -> PIL로 표시 ==========
    # cv.imread()로 읽기 -> PIL로 표시) 색반전 변환 필요 : BGR -> RGB
    # image = Image.fromarray(cv2.cvtColor(image_array.copy(), cv2.COLOR_BGR2RGB))
    
    #========== image array -> CV ==========
    # gradio 컴포넌트로 입력 받을 경우 = array형태로 입력 받음(RGB) -> 변환해줄 필요 없음
    image = Image.fromarray(image_array.copy())
    

    # 그리기 객체 생성
    draw = ImageDraw.Draw(image)

    #========== 이미지를 Yolo 입력용 blob으로 변환 ==========
    blob = cv2.dnn.blobFromImage(image_array, 1/255.0, (416, 416), swapRB=True, crop=False)

    #========== 모델에 넣어주기 ==========
    net.setInput(blob)

    #========== Yolo 실행 ==========
    # unconnected layers : 최종 출력 레이어 지정
    out_layer_list = net.getUnconnectedOutLayersNames()

    # 저장한 출력 레이어까지 순전파(forward pass) 실행
    ## 출력 레이어 3개 -> 결과도 3개(82, 94, 106)
    detection_list = net.forward(out_layer_list) 

    # 모델별로 다른 색상 지정
    color_list = ["red", "green", "blue"] 

    #========== overlap Box 정보 선언 ==========
    overlap_bounding_box_list = list()
    overlap_score_list = list()
    overlap_label_list = list()

    for prediction_list in detection_list:
        color = color_list.pop()

        for prediction in prediction_list:
            
            center_x, center_y, w, h = prediction[:4]  * np.array(
                [image.width, image.height, image.width, image.height]
            )
            # 예측 array 4 : confidence 값
            confidence = prediction[4]
            # 예측 array 5~84 : 각 라벨에 대한 확률값
            score_list = prediction[5:]

            #========== 박스 시작점(왼쪽위) 좌표 계산 ==========
            # 중간 좌표 -> 시작 좌표
            # 음수값이 되는 걸 방지하기 위해 max 적용
            x = max(0, center_x - w/2)  
            y = max(0, center_y - h/2)

            label_index = np.argmax(score_list)
            max_score = score_list[label_index]

            label_text= label_list[label_index]
            if max_score > 0 :
                #========== nms 처리할 변수에 일단 저장 ==========
                overlap_bounding_box_list.append([x,y,w,h])
                overlap_label_list.append(label_text)
                overlap_score_list.append(max_score)

    #========== nms박스들만 인덱스 저장 ==========      
    extracted_index_list = cv2.dnn.NMSBoxes(
                    bboxes= overlap_bounding_box_list, 
                    scores= overlap_score_list, 
                    score_threshold= 0.3, 
                    nms_threshold= 0.4)

    #========== 중복 제거하고 남은 박스만 그리기 ==========
    for extracted_index in extracted_index_list:
        label = overlap_label_list[extracted_index]
        score = overlap_score_list[extracted_index]
        extracted_bounding_box = overlap_bounding_box_list[extracted_index]
        x, y, w, h = extracted_bounding_box

        draw.rectangle( [ (x,y), (x+w, y+h)], outline='green', width= 3)
        draw.text((x,y), "{}({:.2f}%)".format(label, score*100), fill='green')
        # print("{}({:.2f}%)".format(label, score*100))
    return image

#========== 테스트 ==========
# image_array = cv2.imread('sample2.jpg')
# detect_objects(image_array)

In [21]:
# 밑에서 한번에 작성하기 위해 테스트 후 주석처리
# import gradio as gr

# #========== Gradio 화면 ==========

# with gr.Blocks() as demo:
    
#     #========== 이벤트 정의 ==========
#     def stream_webcam(image_array):
#         result_image = detect_objects(image_array)
#         return result_image

#     #========== 컴포넌트 ==========
#     with gr.Row():
#         stream_image = gr.Image(
#             label="웹캠",
#             sources='webcam',
#             # OBS 스튜디오 사용하는 경우
#             webcam_options=gr.WebcamOptions(mirror=False),
#         )
#         result_image = gr.Image(
#             label="감지 결과",
#             interactive=False,
#             type='pil'
#         )
    
#     #========== 이벤트 트리거 ==========
#     stream_image.stream(stream_webcam, inputs=[stream_image], outputs=[result_image])

# demo.launch()

## OpenAI

In [27]:
import gradio as gr
import requests
import cv2
import base64
import os
import dotenv
dotenv.load_dotenv()
YOLO_OPENAI_ENDPOINT = os.getenv('YOLO_OPENAI_ENDPOINT')
YOLO_OPENAI_KEY = os.getenv('YOLO_OPENAI_KEY')
import datetime

#========== 요청 ==========
def request_openai(image_array):
    #========== 이미지 테스트 ==========
    # 이미지를 JPEG로 인코딩, 버퍼에 저장
    _, buffer = cv2.imencode(".jpg", image_array)
    # base64로 인코딩
    image_base64 = base64.b64encode(buffer).decode("utf-8")


    #========== 요청 구성 ==========
    endpoint = YOLO_OPENAI_ENDPOINT

    headers = {
        "Api-key": YOLO_OPENAI_KEY,
        "Content-Type": "application/json"
    }

    body = {
        "messages": [
            {
                "role": "system",
                "content": [
                    {
                        "type": "text",
                        "text": """
                        ### 역할
                        너는 사진 속에서 감지된 물체를 분석하는 AI 봇
                        ### 지침
                        - 분석 결과는 한국어로 답변하세요.
                        - 서론, 결론 제외하고 분석 결과만 답변하세요.
                    """,
                    }
                ],
            },
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": """
                    감지된 물체에 대해서 감지확률과 자세한 설명을 붙여줘.
                    반드시 바운딩 박스로 감지된 물체에 대해서만 설명해야 해.
                    """,
                    },
                    {
                        "type": "image_url",
                        "image_url": {"url": f"data:image/jpeg;base64,{image_base64}"},
                    },
                ],
            },
        ],
        "max_tokens": 4096,
        "temperature": 0.7,
        "top_p": 0.95,
    }


    response = requests.post(endpoint, headers=headers, json=body)
    # response.text 

    # 예외처리
    if not response.ok:
        # pass
        return None

    response_json = response.json()
    role = response_json["choices"][0]["message"]["role"]
    content = response_json["choices"][0]["message"]["content"]

    return {"role": role, "content": content}

#========== 테스트 ==========
# image_array = cv2.imread("sample2.jpg")
# request_openai(image_array)


# 밑에서 한번에 작성하기 위해 테스트 후 주석처리
# #========== Gradio ==========
# with gr.Blocks() as demo:

#     #========== 이벤트 ==========
#     def click_openai_send(image_array, histories):
#         result = request_openai(image_array)
#         label_text = datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
#         histories.append(
#             {
#                 "role": "user",
#                 "content": gr.Image(label=label_text, value=image_array)
#             }
#         )
#         histories.append(result)
#         print(histories)
#         return histories
    
#     #========== Gradio 컴포넌트 ==========
#     capture_image = gr.Image(label="캡쳐 화면")
#     send_openai_button = gr.Button("전송")
#     chatbot = gr.Chatbot(label="분석 결과")
    

#     #========== 이벤트 트리거 ==========
#     send_openai_button.click(click_openai_send, 
#                             inputs=[capture_image, chatbot],
#                             outputs=[chatbot])
# demo.launch()

## TTS

In [ ]:
import gradio as gr
import requests
import os
import dotenv
dotenv.load_dotenv()
YOLO_TTS_ENDPOINT = os.getenv('YOLO_TTS_ENDPOINT')
YOLO_TTS_KEY = os.getenv('YOLO_TTS_KEY')
from IPython.display import Audio
import datetime


#========== 요청 ==========
def request_tts(text):
    endpoint = YOLO_TTS_ENDPOINT

    headers = {
        "X-Microsoft-OutputFormat": "riff-24khz-16bit-mono-pcm",
        "Content-Type": "application/ssml+xml",
        "Ocp-Apim-Subscription-Key": YOLO_TTS_KEY
    }

    body = f"""
        <speak version='1.0' xml:lang='ko-KR'>
            <voice xml:lang='ko-KR' xml:gender='Female' 
            name='ko-KR-SunHi:DragonHDLatestNeural'>
            {text}
            </voice>
        </speak>
    """

    response = requests.post(endpoint, headers=headers, data=body)
    if not response.ok:
        # pass
        return None

    #========== 오디오 파일 저장 ==========
    file_name = "tts_result_{}.wav".format(datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
    file_path = os.path.join("./tts-file", file_name)
    with open(file_path, "wb") as audio_file:
        audio_file.write(response.content)

    return file_path

#========== 테스트 ==========
# print(response)
# display(Audio(response.content))
# request_tts("오늘은 4월 2일 목요일입니다.")


# 밑에서 한번에 작성하기 위해 테스트 후 주석처리
# #========== Gradio 화면 구성 ==========
# with gr.Blocks() as demo:

#     #========== 이벤트 ==========
#     def click_tts_send(text):
#         file_path = request_tts(text)
#         return file_path

#     #========== 컴포넌트 ==========
#     # 나중에는 이미지로 대체, 테스트를 위해 작성
#     input_text = gr.Textbox(label="텍스트 입력")
#     send_tts_button = gr.Button("전송")
#     tts_audio = gr.Audio(label="음성", interactive=False, autoplay=True)

#     #========== 이벤트 트리거 ==========
#     send_tts_button.click(click_tts_send, inputs=[input_text], outputs=[tts_audio])

# demo.launch()

## 통합 관제 시스템

In [ ]:
import datetime

#========== Gradio ==========
with gr.Blocks() as demo:
    
    #========== 이벤트 ==========
    def stream_webcam(image_array):
        result_image = detect_objects(image_array)
        return result_image
          
    def click_capture(image):  
        return image

    # 캡쳐된 이미지가 보인 다음 챗봇이 돌아가도록 capture_image에 이벤트
    def change_capture_image(image, histories):
        #========== 이미지 변환 ==========
        # OpenAI에 던지기 위해 PIL-> numpy array형태로 변환 필요

        # 방법1. 이미지를 단순히 numpy 배열로 강제 변환
        image_array = np.array(image)

        # 방법2. request_openai 함수 수정 -> 이미지 타입에 따른 분기 처리

        #---------- 요청 ----------
        result = request_openai(image_array)
        print(result)
        label_text = datetime.datetime.now().strftime("%Y-%m-%d_%H:%M:%S")
        histories.append(
            {
                "role": "user",
                "content": gr.Image(label=label_text, value=image) #알아서 PIL로 판단해서 보여줌
            }
        )
        histories.append(result)
        print(histories)
        return histories
    
    def change_chatbot(histories):
        # 최초 실행시 예외 처리 (이전 히스토리가 존재하지 않으므로)
        if histories is None or len(histories) == 0:
            return None
        text = histories[-1]['content'][0]['text']
        audio_file_path = request_tts(text)
        return audio_file_path
         

    #========== 컴포넌트 ==========
    gr.Markdown("# 통합 관제 시스템")

    #---------- Yolo ----------
    with gr.Row():
            stream_image = gr.Image(
                label="웹캠",
                sources='webcam',
                # OBS 스튜디오 사용하는 경우
                webcam_options=gr.WebcamOptions(mirror=False),
            )
            result_image = gr.Image(label="감지 결과", interactive=False, type='pil')
            #추가
            capture_image = gr.Image(label='이상 징후 발생', interactive= False)
    #추가    
    capture_button = gr.Button("이상 징후 발생")

    #---------- 챗봇 ----------
    chatbot = gr.Chatbot(label="분석 결과")

    #---------- TTS ----------
    tts_audio = gr.Audio(label="음성", interactive=False, autoplay=True)


    #========== 이벤트 트리거 ==========
    stream_image.stream(stream_webcam, inputs=[stream_image], outputs=[result_image])
    capture_button.click(click_capture, inputs=[result_image], outputs=[capture_image])
    capture_image.change(change_capture_image, inputs=[capture_image, chatbot], outputs=[chatbot])
    chatbot.change(change_chatbot, inputs=[chatbot], outputs=[tts_audio])

demo.launch()


* Running on local URL:  http://127.0.0.1:7872
* To create a public link, set `share=True` in `launch()`.


{'role': 'assistant', 'content': '1. 감지확률: 0.87  \n   바운딩 박스 내 물체: 사람  \n   설명: 왼쪽 바운딩 박스에는 수영장 옆에 서 있는 사람이 감지되었습니다. 이 사람은 파란색 반바지와 어두운 상의를 입고 있으며, 두 손을 모아 입 근처에 가져다 대고 있습니다.\n\n2. 감지확률: 0.92  \n   바운딩 박스 내 물체: 파라솔  \n   설명: 중앙 오른쪽의 바운딩 박스에는 수영장 파라솔이 감지되었습니다. 하얀색과 갈색의 원형 디자인이며, 파라솔 기둥이 아래로 연결되어 있습니다.\n\n3. 감지확률: 0.89  \n   바운딩 박스 내 물체: 사람  \n   설명: 오른쪽 바운딩 박스에는 수영장 옆에 서 있는 또 다른 사람이 감지되었습니다. 이 사람은 밝은 색상의 수영복을 입고 있으며, 한쪽 팔을 들어 올린 자세를 취하고 있습니다.'}
[{'role': 'user', 'content': <gradio.components.image.Image object at 0x0000027B7ED9D150>}, {'role': 'assistant', 'content': '1. 감지확률: 0.87  \n   바운딩 박스 내 물체: 사람  \n   설명: 왼쪽 바운딩 박스에는 수영장 옆에 서 있는 사람이 감지되었습니다. 이 사람은 파란색 반바지와 어두운 상의를 입고 있으며, 두 손을 모아 입 근처에 가져다 대고 있습니다.\n\n2. 감지확률: 0.92  \n   바운딩 박스 내 물체: 파라솔  \n   설명: 중앙 오른쪽의 바운딩 박스에는 수영장 파라솔이 감지되었습니다. 하얀색과 갈색의 원형 디자인이며, 파라솔 기둥이 아래로 연결되어 있습니다.\n\n3. 감지확률: 0.89  \n   바운딩 박스 내 물체: 사람  \n   설명: 오른쪽 바운딩 박스에는 수영장 옆에 서 있는 또 다른 사람이 감지되었습니다. 이 사람은 밝은 색상의 수영복을 입고 있으며, 한쪽 팔을 들어 올린 자세를 취하고 있습니다.'}]
{'role': 'a